In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

In [ ]:
#ω_a = 28.25396662345805
ω_a = 25.338776456203686

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####
#ω_a = 3.85*(rand(1)[1]/5+0.9)*2*pi
ω_b_0 = 2*ω_a

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.9
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0;#3e-3*κ_2 #from Leon's paper

In [ ]:
κ_a_list = 10.0 .^range(start = -2.601893187, stop = -2.6018931743, length = 30)


t_max = 600/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)

ϵ_0 = 0.62
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d = 10.12*g2
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-5,5,10)
Final_all_a = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for κ_a in κ_a_list
    environment = MeanField(κ_a, κ_b)
    Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)
            environment_prepared = prepare(system,environment);

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, _, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            mask  = (real.(a) .>2) .& (imag.(a) .>2)

            Final_mode_a[(re_i,im_j)] = a[mask]
        end
    end
    Final_all_a[κ_a] = Final_mode_a
end

In [ ]:
for κ_a in κ_a_list

    Final_mode = Final_all_a[κ_a]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        z = Final_mode[(re_i, im_j)]
        idx_0, idx_1, idx_2, idx_3 = 4:4:length(z), 1:4:length(z), 2:4:length(z), 3:4:length(z)
        
        scatter!(p0, real.(z[idx_0]), imag.(z[idx_0]), ms = 1, color = :blue, label = "")

        scatter!(p0, real.(z[idx_1]), imag.(z[idx_1]), ms = 1, color = :green, label = "")

        scatter!(p0, real.(z[idx_2]), imag.(z[idx_2]), ms = 1, color = :red, label = "")

        scatter!(p0, real.(z[idx_3]), imag.(z[idx_3]), ms = 1, color = :orange, label = "")
    end

    # Add labels and title. 
    plot!(p0, xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=L"\log_{10}(\kappa_a)="*string(round(log10(κ_a),digits=10)), top_margin=5mm)
    display(p0)
end
